# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [2]:
# Write your code below.
%load_ext dotenv
%dotenv


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
os.getenv("PRICE_DATA")


'../../05_src/data/prices/'

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
import os
import pandas as pd
import dask.dataframe as dd
from glob import glob

# Load data
price_data_dir = os.getenv("PRICE_DATA")
if not price_data_dir:
    raise ValueError("PRICE_DATA environment variable is not set.")

parquet_files = glob(os.path.join(price_data_dir, "**/*.parquet"), recursive=True)
if not parquet_files:
    raise ValueError("No Parquet files found.")

dd_px = dd.read_parquet(parquet_files).set_index("Ticker")




+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
#Convert to Pandas DataFrame
df_px = dd_px.compute()

# Debugging: Print available columns
print("Available columns in the dataset:", df_px.columns)

# Check if 'returns' exists
if 'returns' not in df_px.columns:
    raise ValueError(f"'returns' column is missing in the dataset. Available columns: {df_px.columns}")

# Add a new column with a 10-day moving average of 'returns'
df_px['returns_10d_ma'] = df_px['returns'].rolling(10).mean()

# Display the first few rows
print(df_px.head())

# Note: Can you please help me understand error about return column?


Available columns in the dataset: Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Year'], dtype='object', name='Price')


ValueError: 'returns' column is missing in the dataset. Available columns: Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Year'], dtype='object', name='Price')

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
No, it was not strictly necessary. Dask supports .rolling().mean() operations, allowing the calculation of a moving average without converting the entire dataset to Pandas. However, Dask applies .rolling() only along known divisions (i.e., partitioned data), which may require additional adjustments. 

+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.